# Fine-Tune DistilBERT on RavenPack Headlines (TRNA Substitute)

**Goal:** adapt the PhraseBank-trained DistilBERT classifier to RavenPack news headlines, using RavenPack `event_sentiment_score` as the supervision signal (our TRNA substitute).

Companion module: `src/sentiment_ltr/models/ravenpack_sentiment.py`  
Plan reference: `docs/news_sentiment_finetuning_plan.md` (Iteration 4)

---

## Inputs

| Input | Location / detail |
| --- | --- |
| **RavenPack article export** | `data/raw/news/ravenpack/{ticker}_articles_2003_2014.parquet` — built by `notebooks/fetch_news_articles.ipynb` |
| **Required columns** | `headline`, `event_sentiment_score`, `article_date` (or `article_time`) |
| **Starting checkpoint** (recommended) | `data/models/phrasebank_distilbert_best/` — DistilBERT fine-tuned on Financial PhraseBank |
| **Fallback init** | `distilbert-base-uncased` if no PhraseBank checkpoint exists |
| **Label mapping** | `event_sentiment_score` → `negative` / `neutral` / `positive` (threshold ±0.05) |
| **Train / val / test split** | Time-based: train ≤ 2011, validation = 2012, test ≥ 2013 |
| **Hyperparameters** | `lr=2e-5`, `batch_size=16`, `max_length=128`, `epochs=2` (default) |

**Prerequisites:** conda env `sentiment-ltr-paper` with `requirements-finetuning.txt` installed; at least one ticker with a rich RavenPack export (currently **AAPL**: ~69k labeled headlines).

---

## Steps

1. **Environment check** — confirm `torch`, `transformers`, `datasets`, and GPU/MPS device.
2. **Discover data** — list `*_articles_*.parquet` under `data/raw/news/ravenpack/`; pick ticker(s).
3. **Load & inspect** — labeled rows (headline + score), class balance, split row counts.
4. **Build HF dataset** — map headlines to `sentence`, scores to 3-class `label`; time-based splits.
5. **Load init checkpoint** — PhraseBank weights (recommended) or `distilbert-base-uncased`.
6. **Tokenize** — `max_length=128`, fixed padding (same as PhraseBank pipeline).
7. **Train** — Hugging Face `Trainer`, 2 epochs, eval each epoch, `load_best_model_at_end` on **validation macro-F1**.
8. **Evaluate** — report validation + **test** accuracy and macro-F1.
9. **Save** — write checkpoint + `metrics.json` for the app / batch scoring.

---

## Expected outputs

| Output | Path / description |
| --- | --- |
| **Fine-tuned model** | `data/models/ravenpack_distilbert_best/` (`config.json`, tokenizer, weights) |
| **Training metrics** | `data/models/ravenpack_distilbert_best/metrics.json` — train loss, val/test `eval_f1`, `eval_accuracy`, hyperparams, tickers used |
| **Console summary** | Split sizes (e.g. AAPL: ~38k train / ~13k val / ~18k test), class balance, final test macro-F1 |
| **Downstream use** | Sentiment Lab → *Fine-tune on RavenPack headlines*; future batch-scoring of cached headlines (Iteration 4.2) |

**Success criteria:** training completes without error; test macro-F1 is reported; checkpoint loads for inference on new headlines.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

%load_ext autoreload
%autoreload 2

from sentiment_ltr.models.phrasebank_sentiment import (
    DEFAULT_MODEL_DIR,
    label_maps,
    load_classifier,
    load_phrasebank,
    model_is_saved,
    predict_sentences,
)

N = 2  # samples per split

# 1. Load PhraseBank-trained checkpoint
if not model_is_saved(DEFAULT_MODEL_DIR):
    raise FileNotFoundError(
        f"No checkpoint at {DEFAULT_MODEL_DIR}. Run notebooks/liquidAI_prep.ipynb first."
    )
tokenizer, model, device = load_classifier(DEFAULT_MODEL_DIR)

# 2. Sample inferences on PhraseBank validation & test (splits the model was trained on)
raw = load_phrasebank()
_, id2label, _ = label_maps(raw)

val = raw["validation"].shuffle(seed=42).select(range(N))
test = raw["test"].shuffle(seed=42).select(range(N))

samples = pd.concat(
    [
        pd.DataFrame({
            "split": "validation",
            "sentence": val["sentence"],
            "label": [id2label[int(i)] for i in val["label"]],
        }),
        pd.DataFrame({
            "split": "test",
            "sentence": test["sentence"],
            "label": [id2label[int(i)] for i in test["label"]],
        }),
    ],
    ignore_index=True,
)

preds = predict_sentences(samples["sentence"].tolist(), tokenizer, model, device)
samples.join(preds.drop(columns=["sentence"])).assign(
    match=lambda df: df["label"] == df["pred"]
)


In [ ]:
from sentiment_ltr.models.phrasebank_sentiment import phrasebank_probability_chart_frame
from sentiment_ltr.viz import split_series_distribution_figures

CLASS_ORDER = ["negative", "neutral", "positive"]

long_probs = phrasebank_probability_chart_frame()
split_order = long_probs["split"].cat.categories.tolist()
chart_orders = {"split": split_order, "class": CLASS_ORDER}
chart_labels = {
    "split": "PhraseBank split",
    "probability": "Predicted probability",
    "p50": "Median predicted probability",
    "class": "Class",
}

fig_box, fig_p50, p50 = split_series_distribution_figures(
    long_probs,
    x="split",
    y="probability",
    series="class",
    category_orders=chart_orders,
    axis_labels=chart_labels,
    box_title="Class probabilities by split (box & whisker)",
    median_title="Median class probability by split (p50)",
    median_col="p50",
    x_hover_label="PhraseBank split",
    series_hover_label="Class",
    y_hover_label="Predicted probability",
    median_y_hover_label="Median predicted probability",
)
fig_box.show()
fig_p50.show()
